In [1]:
# Importing libraries
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [2]:
# Loading the cleaned files
current_path = Path.cwd()

if current_path.name == 'notebooks':
    project_path = current_path.parent
else:
    project_path = current_path

data_path = project_path / 'data' / 'processed'

if not data_path.exists():
    project_path = current_path.parent
    data_path = project_path / 'data' / 'processed'

customers = pd.read_csv(
    data_path / 'customers.csv',
    parse_dates=['signup_date']
)

memberships = pd.read_csv(
    data_path / 'memberships.csv',
    parse_dates=[
        'membership_start_date',
        'renewal_date',
        'membership_end_date',
        'cancellation_date'
    ]
)

orders = pd.read_csv(
    data_path / 'orders.csv',
    parse_dates=['order_date']
)

prime_video = pd.read_csv(
    data_path / 'prime_video_activity.csv',
    parse_dates=['activity_date']
)

payments = pd.read_csv(
    data_path / 'payments.csv',
    parse_dates=['payment_date']
)

support = pd.read_csv(
    data_path / 'support_interactions.csv',
    parse_dates=['ticket_date']
)

print('Customers:', len(customers))
print('Memberships:', len(memberships))
print('Orders:', len(orders))
print('Prime Video:', len(prime_video))
print('Payments:', len(payments))
print('Support:', len(support))


Customers: 50000
Memberships: 55000
Orders: 399976
Prime Video: 549970
Payments: 124984
Support: 34990


In [3]:
# Setting the model dates
scoring_date = pd.Timestamp('2026-05-31')

feature_30_start = scoring_date - pd.Timedelta(days=29)
feature_90_start = scoring_date - pd.Timedelta(days=89)

prediction_start = scoring_date + pd.Timedelta(days=1)
prediction_end = scoring_date + pd.Timedelta(days=30)

print('30-day feature window:', feature_30_start.date(), 'to', scoring_date.date())
print('90-day feature window:', feature_90_start.date(), 'to', scoring_date.date())
print('Prediction window:', prediction_start.date(), 'to', prediction_end.date())


30-day feature window: 2026-05-02 to 2026-05-31
90-day feature window: 2026-03-03 to 2026-05-31
Prediction window: 2026-06-01 to 2026-06-30


In [4]:
# Selecting customers who were active on the scoring date
membership_history = memberships.sort_values(
    ['customer_id', 'membership_start_date', 'membership_id']
).copy()

membership_history['next_membership_start_date'] = (
    membership_history.groupby('customer_id')['membership_start_date'].shift(-1)
)

latest_membership = membership_history[
    membership_history['membership_start_date'] <= scoring_date
].sort_values(
    ['customer_id', 'membership_start_date', 'renewal_date', 'membership_id']
).groupby(
    'customer_id',
    as_index=False
).tail(1).copy()

active_at_scoring = (
    (
        latest_membership['membership_end_date'].isna()
        | (latest_membership['membership_end_date'] > scoring_date)
    )
    & (
        latest_membership['cancellation_date'].isna()
        | (latest_membership['cancellation_date'] > scoring_date)
    )
)

model_memberships = latest_membership[active_at_scoring].copy()

model_memberships = model_memberships[
    ~model_memberships['membership_status'].isin(
        ['Paused', 'Payment Pending']
    )
].copy()

uncertain_expiry = (
    model_memberships['membership_status'].eq('Expired')
    & model_memberships['membership_end_date'].between(
        '2026-06-24',
        '2026-06-30'
    )
)

model_memberships = model_memberships[~uncertain_expiry].copy()

print('Customers eligible for prediction:', len(model_memberships))


Customers eligible for prediction: 35193


In [5]:
# Creating the 30-day churn target
cancelled_in_window = (
    model_memberships['membership_status'].eq('Cancelled')
    & model_memberships['cancellation_date'].between(
        prediction_start,
        prediction_end
    )
    & (
        model_memberships['next_membership_start_date'].isna()
        | (
            model_memberships['next_membership_start_date']
            > prediction_end
        )
    )
)

expired_in_window = (
    model_memberships['membership_status'].eq('Expired')
    & model_memberships['membership_end_date'].between(
        prediction_start,
        pd.Timestamp('2026-06-23')
    )
    & ~model_memberships['next_membership_start_date'].between(
        model_memberships['membership_end_date'],
        model_memberships['membership_end_date']
        + pd.Timedelta(days=7)
    )
)

model_memberships['churn_next_30_days'] = (
    cancelled_in_window | expired_in_window
).astype(int)

model_memberships['tenure_months'] = (
    (
        scoring_date
        - model_memberships['membership_start_date']
    ).dt.days / 30.44
).clip(lower=0).round(1)

model_memberships['auto_renew_enabled'] = (
    model_memberships['auto_renew_enabled']
    .map({True: 'Yes', False: 'No'})
    .fillna('Unknown')
)

model_memberships = model_memberships[
    [
        'customer_id',
        'tenure_months',
        'plan_type',
        'auto_renew_enabled',
        'discount_applied',
        'billing_cycle',
        'churn_next_30_days'
    ]
].copy()

print(model_memberships['churn_next_30_days'].value_counts())


churn_next_30_days
0    34210
1      983
Name: count, dtype: int64


In [6]:
# Starting the customer-level model dataset
model_data = customers[
    [
        'customer_id',
        'country',
        'age_group',
        'acquisition_channel',
        'primary_device'
    ]
].merge(
    model_memberships,
    on='customer_id',
    how='inner'
)

print('Model rows:', len(model_data))


Model rows: 35193


In [7]:
# Creating shopping features
valid_orders = orders[
    (orders['order_date'] <= scoring_date)
    & orders['order_status'].ne('Cancelled')
].copy()

orders_30 = valid_orders[
    valid_orders['order_date'].between(
        feature_30_start,
        scoring_date
    )
].copy()

orders_90 = valid_orders[
    valid_orders['order_date'].between(
        feature_90_start,
        scoring_date
    )
].copy()

orders_30_summary = orders_30.groupby('customer_id').agg(
    orders_last_30_days=('order_id', 'count')
).reset_index()

orders_90_summary = orders_90.groupby('customer_id').agg(
    orders_last_90_days=('order_id', 'count'),
    spend_last_90_days=('order_value', 'sum'),
    average_order_value=('order_value', 'mean'),
    shipping_fee_saved=('shipping_fee_saved', 'sum')
).reset_index()

delivered_90 = orders_90[
    orders_90['order_status'].eq('Delivered')
].groupby('customer_id').agg(
    delivered_orders=('order_id', 'count'),
    late_deliveries=('delivered_late_flag', 'sum')
).reset_index()

returned_90 = orders_90[
    orders_90['order_status'].isin(['Delivered', 'Returned'])
].groupby('customer_id').agg(
    return_eligible_orders=('order_id', 'count'),
    returned_orders=('returned_flag', 'sum')
).reset_index()

last_order = valid_orders.groupby('customer_id').agg(
    last_order_date=('order_date', 'max')
).reset_index()

model_data = model_data.merge(
    orders_30_summary,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    orders_90_summary,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    delivered_90,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    returned_90,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    last_order,
    on='customer_id',
    how='left'
)

model_data['late_delivery_rate'] = (
    model_data['late_deliveries']
    / model_data['delivered_orders']
)

model_data['return_rate'] = (
    model_data['returned_orders']
    / model_data['return_eligible_orders']
)

model_data['days_since_last_order'] = (
    scoring_date - model_data['last_order_date']
).dt.days


In [8]:
# Creating Prime Video features
video_before_score = prime_video[
    prime_video['activity_date'] <= scoring_date
].copy()

video_30 = video_before_score[
    video_before_score['activity_date'].between(
        feature_30_start,
        scoring_date
    )
].copy()

video_90 = video_before_score[
    video_before_score['activity_date'].between(
        feature_90_start,
        scoring_date
    )
].copy()

video_30_summary = video_30.groupby('customer_id').agg(
    watch_minutes_last_30_days=('watch_minutes', 'sum')
).reset_index()

video_90_summary = video_90.groupby('customer_id').agg(
    watch_minutes_last_90_days=('watch_minutes', 'sum'),
    video_sessions_last_90_days=('sessions_count', 'sum'),
    average_completion_rate=('completion_rate', 'mean')
).reset_index()

last_video = video_before_score.groupby('customer_id').agg(
    last_video_activity_date=('activity_date', 'max')
).reset_index()

model_data = model_data.merge(
    video_30_summary,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    video_90_summary,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    last_video,
    on='customer_id',
    how='left'
)

model_data['days_since_last_video_activity'] = (
    scoring_date - model_data['last_video_activity_date']
).dt.days


In [9]:
# Creating payment features
payments_before_score = payments[
    payments['payment_date'] <= scoring_date
].copy()

payments_90 = payments_before_score[
    payments_before_score['payment_date'].between(
        feature_90_start,
        scoring_date
    )
].copy()

payments_90['payment_failed_flag'] = (
    payments_90['payment_status'].eq('Failed').astype(int)
)

payment_summary = payments_90.groupby('customer_id').agg(
    payment_failures_last_90_days=('payment_failed_flag', 'sum'),
    payment_retries_last_90_days=('retry_count', 'sum')
).reset_index()

last_payment = payments_before_score.sort_values(
    ['customer_id', 'payment_date', 'payment_id']
).groupby(
    'customer_id',
    as_index=False
).tail(1).copy()

last_payment['last_payment_failed'] = (
    last_payment['payment_status'].eq('Failed').astype(int)
)

last_payment = last_payment[
    ['customer_id', 'last_payment_failed']
]

model_data = model_data.merge(
    payment_summary,
    on='customer_id',
    how='left'
)

model_data = model_data.merge(
    last_payment,
    on='customer_id',
    how='left'
)


In [10]:
# Creating support features
support_90 = support[
    support['ticket_date'].between(
        feature_90_start,
        scoring_date
    )
].copy()

support_summary = support_90.groupby('customer_id').agg(
    support_tickets_last_90_days=('ticket_id', 'count'),
    average_resolution_hours=('resolution_hours', 'mean'),
    average_satisfaction_score=('satisfaction_score', 'mean'),
    repeat_contact_rate=('repeat_contact_flag', 'mean')
).reset_index()

model_data = model_data.merge(
    support_summary,
    on='customer_id',
    how='left'
)


In [11]:
# Creating combined engagement features
order_days = orders_90[
    ['customer_id', 'order_date']
].rename(
    columns={'order_date': 'active_date'}
)

video_days = video_90[
    ['customer_id', 'activity_date']
].rename(
    columns={'activity_date': 'active_date'}
)

active_days = pd.concat(
    [order_days, video_days],
    ignore_index=True
).drop_duplicates()

active_days = active_days.groupby('customer_id').size().reset_index(
    name='total_active_days'
)

model_data = model_data.merge(
    active_days,
    on='customer_id',
    how='left'
)

zero_columns = [
    'orders_last_30_days',
    'orders_last_90_days',
    'spend_last_90_days',
    'average_order_value',
    'shipping_fee_saved',
    'delivered_orders',
    'late_deliveries',
    'return_eligible_orders',
    'returned_orders',
    'watch_minutes_last_30_days',
    'watch_minutes_last_90_days',
    'video_sessions_last_90_days',
    'average_completion_rate',
    'payment_failures_last_90_days',
    'payment_retries_last_90_days',
    'last_payment_failed',
    'support_tickets_last_90_days',
    'repeat_contact_rate',
    'total_active_days'
]

model_data[zero_columns] = model_data[zero_columns].fillna(0)

model_data['late_delivery_rate'] = (
    model_data['late_delivery_rate'].fillna(0)
)

model_data['return_rate'] = (
    model_data['return_rate'].fillna(0)
)

model_data['days_since_last_order'] = (
    model_data['days_since_last_order'].fillna(999).astype(int)
)

model_data['days_since_last_video_activity'] = (
    model_data['days_since_last_video_activity']
    .fillna(999)
    .astype(int)
)

model_data['days_since_last_activity'] = model_data[
    [
        'days_since_last_order',
        'days_since_last_video_activity'
    ]
].min(axis=1)

discount_used = ~model_data['discount_applied'].isin(
    ['No Discount', 'Not Applicable', 'Unknown']
)

model_data['benefits_used_count'] = (
    model_data['orders_last_90_days'].gt(0).astype(int)
    + model_data['shipping_fee_saved'].gt(0).astype(int)
    + model_data['watch_minutes_last_90_days'].gt(0).astype(int)
    + discount_used.astype(int)
)

model_data['engagement_score'] = (
    np.minimum(model_data['orders_last_90_days'] * 3, 20)
    + np.minimum(
        model_data['watch_minutes_last_90_days'] / 60,
        20
    )
    + np.minimum(
        model_data['video_sessions_last_90_days'],
        20
    )
    + np.minimum(model_data['total_active_days'], 20)
    + model_data['benefits_used_count'] * 5
).round(2)


In [12]:
# Creating a customer segment using pre-scoring data
segment_conditions = [
    (
        model_data['payment_failures_last_90_days'].gt(0)
        | model_data['payment_retries_last_90_days'].ge(2)
        | (
            model_data['auto_renew_enabled'].eq('No')
            & model_data['payment_retries_last_90_days'].gt(0)
        )
    ),
    (
        model_data['support_tickets_last_90_days'].ge(2)
        | model_data['average_satisfaction_score'].lt(3)
        | model_data['late_delivery_rate'].gt(0.10)
    ),
    model_data['tenure_months'].lt(3),
    (
        model_data['orders_last_90_days'].ge(3)
        & model_data['watch_minutes_last_90_days'].ge(600)
        & model_data['benefits_used_count'].ge(3)
    ),
    (
        model_data['orders_last_90_days'].ge(3)
        & model_data['watch_minutes_last_90_days'].lt(600)
    ),
    (
        model_data['watch_minutes_last_90_days'].ge(600)
        & model_data['orders_last_90_days'].lt(3)
    ),
    (
        model_data['orders_last_90_days'].le(1)
        & model_data['watch_minutes_last_90_days'].lt(120)
        & model_data['days_since_last_activity'].gt(30)
    )
]

segment_names = [
    'Payment-Risk Members',
    'Service-Risk Members',
    'New Members',
    'Multi-Benefit Power Users',
    'Shopping-First Members',
    'Video-First Members',
    'Low-Engagement Members'
]

model_data['customer_segment'] = np.select(
    segment_conditions,
    segment_names,
    default='Regular Members'
)

print(model_data['customer_segment'].value_counts())


customer_segment
Regular Members              9946
Shopping-First Members       6858
New Members                  5565
Service-Risk Members         3877
Video-First Members          3075
Low-Engagement Members       2769
Multi-Benefit Power Users    2077
Payment-Risk Members         1026
Name: count, dtype: int64


In [13]:
# Keeping the final model columns
final_columns = [
    'customer_id',
    'country',
    'age_group',
    'acquisition_channel',
    'primary_device',
    'tenure_months',
    'plan_type',
    'auto_renew_enabled',
    'discount_applied',
    'billing_cycle',
    'orders_last_30_days',
    'orders_last_90_days',
    'spend_last_90_days',
    'average_order_value',
    'days_since_last_order',
    'late_delivery_rate',
    'return_rate',
    'shipping_fee_saved',
    'watch_minutes_last_30_days',
    'watch_minutes_last_90_days',
    'video_sessions_last_90_days',
    'days_since_last_video_activity',
    'average_completion_rate',
    'payment_failures_last_90_days',
    'payment_retries_last_90_days',
    'last_payment_failed',
    'support_tickets_last_90_days',
    'average_resolution_hours',
    'average_satisfaction_score',
    'repeat_contact_rate',
    'benefits_used_count',
    'total_active_days',
    'days_since_last_activity',
    'engagement_score',
    'customer_segment',
    'churn_next_30_days'
]

churn_model_data = model_data[final_columns].copy()

integer_columns = [
    'orders_last_30_days',
    'orders_last_90_days',
    'days_since_last_order',
    'video_sessions_last_90_days',
    'days_since_last_video_activity',
    'payment_failures_last_90_days',
    'payment_retries_last_90_days',
    'last_payment_failed',
    'support_tickets_last_90_days',
    'benefits_used_count',
    'total_active_days',
    'days_since_last_activity',
    'churn_next_30_days'
]

churn_model_data[integer_columns] = (
    churn_model_data[integer_columns].astype(int)
)

decimal_columns = [
    'spend_last_90_days',
    'average_order_value',
    'shipping_fee_saved',
    'watch_minutes_last_30_days',
    'watch_minutes_last_90_days',
    'average_resolution_hours',
    'average_satisfaction_score',
    'engagement_score'
]

churn_model_data[decimal_columns] = (
    churn_model_data[decimal_columns].round(2)
)

rate_columns = [
    'late_delivery_rate',
    'return_rate',
    'average_completion_rate',
    'repeat_contact_rate'
]

churn_model_data[rate_columns] = (
    churn_model_data[rate_columns].round(4)
)


In [14]:
# Checking the final dataset
leakage_columns = [
    'membership_status',
    'cancellation_date',
    'membership_end_date',
    'cancellation_reason',
    'churn_flag',
    'account_status'
]

leakage_found = [
    column
    for column in leakage_columns
    if column in churn_model_data.columns
]

print('Rows:', len(churn_model_data))
print('Columns:', len(churn_model_data.columns))
print('Unique customers:', churn_model_data['customer_id'].nunique())
print('Duplicate customers:', churn_model_data['customer_id'].duplicated().sum())
print('Leakage columns found:', leakage_found)
print()
print('Target distribution:')
print(churn_model_data['churn_next_30_days'].value_counts())
print()
print('Target percentage:')
print(
    churn_model_data['churn_next_30_days']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
print()
print('Missing values:')
print(
    churn_model_data.isna().sum()[
        churn_model_data.isna().sum() > 0
    ]
)


Rows: 35193
Columns: 36
Unique customers: 35193
Duplicate customers: 0
Leakage columns found: []

Target distribution:
churn_next_30_days
0    34210
1      983
Name: count, dtype: int64

Target percentage:
churn_next_30_days
0   97.21
1    2.79
Name: proportion, dtype: float64

Missing values:
average_resolution_hours      29395
average_satisfaction_score    29395
dtype: int64


In [15]:
# Comparing a few features by target
feature_check = churn_model_data.groupby(
    'churn_next_30_days'
)[
    [
        'orders_last_90_days',
        'spend_last_90_days',
        'watch_minutes_last_90_days',
        'video_sessions_last_90_days',
        'payment_failures_last_90_days',
        'support_tickets_last_90_days',
        'average_satisfaction_score',
        'engagement_score'
    ]
].mean().round(2)

feature_check


,orders_last_90_days,spend_last_90_days,watch_minutes_last_90_days,video_sessions_last_90_days,payment_failures_last_90_days,support_tickets_last_90_days,average_satisfaction_score,engagement_score
churn_next_30_days,,,,,,,,
0,2.13,86.71,324.87,5.95,0.03,0.20,3.67,35.98
1,1.02,41.76,172.78,3.23,0.06,0.38,3.30,22.79


In [16]:
# Saving the model dataset
output_file = data_path / 'churn_model_dataset.csv'

churn_model_data.to_csv(
    output_file,
    index=False
)

print('Saved:', output_file)
print('Rows saved:', len(churn_model_data))


Saved: c:\Users\naren\OneDrive\Desktop\Data Analytics\Projects\Portfolio Centerpiece Projects using AI\End-to-End Amazon Prime Customer Churn & Retention Analytics\data\processed\churn_model_dataset.csv
Rows saved: 35193
